# In-Class Exercise: Predicting Car Prices with Regression

In this exercise we use F-150 listings from the car listings data to explore a core idea: **adding information reduces uncertainty**. We start by guessing a price with almost nothing to go on, then refine our guess as we learn more. At the end, we formalize this process with a linear regression model.

**Data**: Use the same `car_listings.csv` from previous assignments. Place it in this folder before running.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Standard cleaning pipeline (same as previous assignments)
# listings = pd.read_csv('car_listings.csv')
listings = pd.read_csv('../../2026/spring/data/car_listings.zip')

listings['time_posted']          = pd.to_datetime(listings['time_posted'], errors='coerce')
listings['year_from_time_posted'] = listings['time_posted'].dt.year
listings['year']       = pd.to_numeric(listings['year'],       errors='coerce').astype('Int64')
listings['odometer']   = pd.to_numeric(listings['odometer'],   errors='coerce').astype('Int64')
listings['post_id']    = pd.to_numeric(listings['post_id'],    errors='coerce').astype('Int64')
listings['num_images'] = pd.to_numeric(listings['num_images'], errors='coerce').astype('Int64')
listings['price']      = pd.to_numeric(listings['price'],      errors='coerce')
listings['latitude']   = pd.to_numeric(listings['latitude'],   errors='coerce')
listings['longitude']  = pd.to_numeric(listings['longitude'],  errors='coerce')

listings = listings.drop_duplicates(subset='post_id')
for col in ['make', 'model', 'location', 'title', 'fuel', 'drive',
            'transmission', 'paint', 'type', 'condition']:
    listings[col] = listings[col].str.lower()

listings['car_age'] = listings['year_from_time_posted'] - listings['year']

print(listings.shape)

In [ ]:
# Filter to F-150s with clean titles and valid numeric data in Minneapolis
f150 = (
    listings
    .query("make == 'ford' and model == 'f150'")
    .query("location == 'minneapolis'")
    .query("title == 'clean'")
    .dropna(subset=['price', 'odometer', 'car_age', 'location'])
    .query('price > 0 and odometer > 0 and car_age >= 0')
    .copy()
)
print(f'F-150 listings: {len(f150)}')

---

## Part 1: Guess the Price

You will be given information describing a specific F-150 for sale in Minneapolis. As you learn more about it, use the `f150` data frame to refine your estimate of the listing price.

Work in the cells below — there's no single right approach. Use whatever summary statistics or visualizations seem useful.

In [ ]:
# Your exploration here on the first question, "How much is an F150 in Minneapolis?"


In [ ]:
# If you know the year, does that change your guess? 


In [ ]:
# What about if you know the mileage?


---

*Pause here for class discussion.*

---

## Part 2: A Regression Model

What we just did by hand — filtering to a year, then to a mileage range, then to a city — is slow and hard to generalize. **Linear regression** automates it: we fit a single equation that uses all of these variables together to minimize prediction error.

We'll model `price` as a function of three features:

| Feature | Description |
|---|---|
| `car_age` | Years since the model year (at time of listing) |
| `log_odometer` | Natural log of odometer reading — compresses the long right tail of mileage |
| `in_minneapolis` | 1 if the listing is from Minneapolis, 0 otherwise |

### 2a. Feature Engineering

`car_age` already exists from the cleaning step. Create the other two features.

In [ ]:
# Natural log of odometer (provided)
f150['log_odometer'] = np.log(f150['odometer'])

# YOUR CODE HERE: create 'in_minneapolis' — 1 if the listing is from Minneapolis, 0 otherwise
# Hint: (f150['location'] == 'minneapolis').astype(int)
f150['in_minneapolis'] = ___

f150[['car_age', 'log_odometer', 'in_minneapolis', 'price']].describe()

### 2b. Fitting the Model

`statsmodels.formula.api` lets us specify models with R-style formula strings. The formula `'price ~ car_age + log_odometer + in_minneapolis'` means: model `price` as a linear function of those three predictors.

Hint: call `smf.ols(formula, data=f150).fit()` and save the result as `results`.

In [ ]:
# YOUR CODE HERE: fit the OLS model and save as results
results = ___

In [ ]:
results.summary()

### 2c. Interpreting the Results

The **coef** column in the summary gives the estimated coefficients. Each coefficient tells you how much the predicted price changes when that variable increases by 1, holding everything else fixed.

Extract the individual coefficients below and answer each question.

**Q1. `car_age` coefficient**

Extract the coefficient and complete the print statement.

Hint: call `.params['car_age']` on `results`.

In [ ]:
# YOUR CODE HERE
coef_age = ___
print(f'Each additional year of age is associated with a ${coef_age:,.0f} change in price.')

*Does the sign (positive or negative) make intuitive sense? Why or why not?*

*Your answer here:*

**Q2. `log_odometer` coefficient**

Extract the coefficient for `log_odometer`.

Hint: call `.params['log_odometer']` on `results`.

In [ ]:
# YOUR CODE HERE
coef_odo = ___
print(f'log_odometer coefficient: {coef_odo:,.0f}')

*Because we used the natural log of odometer, a 1-unit increase in `log_odometer` means multiplying mileage by e (roughly 2.72x) — e.g., going from 50k to 136k miles. In plain English, what does this coefficient tell you?*

*Your answer here:*

**Q3. `in_minneapolis` coefficient**

Extract the coefficient for `in_minneapolis`.

Hint: call `.params['in_minneapolis']` on `results`.

In [ ]:
# YOUR CODE HERE
coef_mpls = ___
print(f'Minneapolis price effect: ${coef_mpls:,.0f}')

*Compared to an otherwise identical listing in another city in the dataset, what does this coefficient say about F-150 prices in Minneapolis?*

*Your answer here:*

**Q4. R-squared**

Look at the `R-squared` value in `results.summary()`. What fraction of the variation in F-150 prices does this model explain? Does that seem like a lot or a little, given what else might affect used car prices?

*Your answer here:*

### 2d. Making a Prediction

We've described a specific F-150. Fill in its features and use the model to predict its price.

Hint: build a one-row DataFrame with the right column names, then call `.predict(target)` on `results`.

In [ ]:
target = pd.DataFrame({
    'car_age':        [___],   # YOUR CODE HERE: how old is this truck?
    'log_odometer':   [___],   # YOUR CODE HERE: np.log(___)
    'in_minneapolis': [1],
})

# YOUR CODE HERE: use results.predict(target) to get the predicted price
predicted_price = ___
print(f'Predicted price: ${predicted_price.iloc[0]:,.0f}')

*Now you'll get the actual listing price. How does the model's prediction compare? Would you call this a good deal, a bad deal, or about what the model expects?*

*Your answer here:*